# Usage of PySpark SQL

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
                     .appName("Analyzing an unknown article.")
                     .getOrCreate())


In [2]:
spark

In [3]:
sc = spark.sparkContext

In [5]:
## documentation

spark.read??

In [8]:
file_path = r'article.txt'

In [9]:
article = spark.read.text(file_path)

In [10]:
article

DataFrame[value: string]

In [11]:
article.printSchema()

root
 |-- value: string (nullable = true)



In [12]:
article.select(article.value)

DataFrame[value: string]

In [13]:
article.show(5, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                                                                                                                                        |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
from pyspark.sql.functions import col

In [15]:

article.select(article.value)
article.select(article['value'])
article.select(col('value'))
article.select('value')

DataFrame[value: string]

In [16]:
from pyspark.sql.functions import col, split

lines = article.select(
    split(col('value'), " ").alias('line')
)

In [17]:
lines.printSchema()

root
 |-- line: array (nullable = true)
 |    |-- element: string (containsNull = false)



In [18]:
lines.show(5, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|line                                                                                                                                                                                                                                                                                                                                                                                                                                                        |
+---------------------------------------------------------------------------------------------------------

In [19]:
lines

DataFrame[line: array<string>]

In [20]:
from pyspark.sql.functions import explode

words = lines.select(explode(col("line")).alias('word'))

In [21]:
words.printSchema()

root
 |-- word: string (nullable = false)



In [22]:
from pyspark.sql.functions import lower

words_lower = words.select(lower(col("word")).alias('word_lower'))

In [23]:
words_lower.show(10)

+---------------+
|     word_lower|
+---------------+
|            the|
|          white|
|          house|
|correspondents'|
|    association|
|         dinner|
|            has|
|           long|
|           been|
|           held|
+---------------+
only showing top 10 rows


In [24]:
from pyspark.sql.functions import regexp_extract

words_clean = words_lower.select(
    regexp_extract(col("word_lower"), r"(\W+)?([a-z]+)", 2).alias("word_clean")
)

In [25]:
words_clean.show(10)

+--------------+
|    word_clean|
+--------------+
|           the|
|         white|
|         house|
|correspondents|
|   association|
|        dinner|
|           has|
|          long|
|          been|
|          held|
+--------------+
only showing top 10 rows


In [26]:
words_nonull = words_clean.where(col("word_clean") != "")

words_nonull.show(100)

+--------------+
|    word_clean|
+--------------+
|           the|
|         white|
|         house|
|correspondents|
|   association|
|        dinner|
|           has|
|          long|
|          been|
|          held|
|            at|
|           the|
|    washington|
|        hilton|
|         which|
|         hosts|
|             a|
|           lot|
|            of|
|          high|
|       profile|
|        events|
|            at|
|         least|
|            in|
|          part|
|       because|
|            of|
|             a|
|        unique|
|        design|
|          that|
|            is|
|  specifically|
|      intended|
|           for|
|  presidential|
|      security|
|         there|
|          even|
|             a|
|       special|
|      entrance|
|           for|
|           the|
|     president|
|           and|
|             a|
|     dedicated|
|       holding|
|          room|
|        behind|
|           the|
|         stage|
|          with|
|             

In [41]:
groups = words_nonull.groupBy(col("word_clean"))

In [42]:
groups

GroupedData[grouping expressions: [word_clean], value: [word_clean: string], type: GroupBy]

In [43]:
counts = groups.count()

In [30]:
counts.orderBy('count', ascending=False).show(10)

+----------+-----+
|word_clean|count|
+----------+-----+
|       the|  107|
|         a|   39|
|       and|   37|
|        of|   29|
| president|   28|
|        to|   27|
|      that|   25|
|        in|   21|
|      this|   19|
|        he|   19|
+----------+-----+
only showing top 10 rows


In [31]:
import pyspark.sql.functions as F

counts = (
    spark.read.text(file_path)
     .select(F.split(F.col('value'), ' ').alias('line'))
     .select(F.explode(F.col('line')).alias('word'))
     .select(F.lower(F.col('word')).alias('word'))
     .select(F.regexp_extract(F.col('word'), r"(\W+)?([a-z]+)", 2).alias('word'))
     .where(F.col('word') != "")
     .groupby('word')
     .count()
)

In [32]:
counts.show(10)

+---------------+-----+
|           word|count|
+---------------+-----+
|          those|    2|
|           some|    1|
|          still|    1|
|          staff|    2|
|   logistically|    1|
|         worked|    1|
|            don|    1|
|         donald|    1|
|          scary|    1|
|extraordinarily|    1|
+---------------+-----+
only showing top 10 rows


In [36]:
words_length = words_clean.filter(F.length(col("word")) > 3)

In [38]:
words_length.show(10)

+--------------+
|    word_clean|
+--------------+
|         white|
|         house|
|correspondents|
|   association|
|        dinner|
|          long|
|          been|
|          held|
|    washington|
|        hilton|
+--------------+
only showing top 10 rows


In [45]:
groups_len = words_length.groupBy(col("word_clean"))
counts_len = groups_len.count()
counts_len.orderBy('count', ascending=False).show(5)

+----------+-----+
|word_clean|count|
+----------+-----+
| president|   28|
|      that|   25|
|      this|   19|
|  security|   18|
|      with|   16|
+----------+-----+
only showing top 5 rows


In [49]:
# word lengths and occurrence frequencies
counts = (
    spark.read.text(file_path)
     .select(F.split(F.col('value'), ' ').alias('line'))
     .select(F.explode(F.col('line')).alias('word'))
     .select(F.lower(F.col('word')).alias('word'))
     .select(F.regexp_extract(F.col('word'), r"(\W+)?([a-z]+)", 2).alias('word'))
     .where(F.col('word') != "")
     .groupby(F.length(col("word")))
     .count()
)
counts.show(5)

+------------+-----+
|length(word)|count|
+------------+-----+
|          12|    7|
|           1|   51|
|          13|    4|
|           6|  124|
|           3|  285|
+------------+-----+
only showing top 5 rows


In [64]:
alphabets = words.select(F.split(F.col('word'), '').alias('letters'))
alphabets.select(F.explode('letters')).show()

+---+
|col|
+---+
|  T|
|  h|
|  e|
|  W|
|  h|
|  i|
|  t|
|  e|
|  H|
|  o|
|  u|
|  s|
|  e|
|  C|
|  o|
|  r|
|  r|
|  e|
|  s|
|  p|
+---+
only showing top 20 rows
